# Google Account Setup

This notebook connects your Google account to the automation service.
It takes about **2 minutes** and you only need to do it once.

**Before you start, make sure you have:**
- The `client_secret.json` file (sent to you separately)

---
▶️ **Run each cell in order by clicking the ▶ button on the left side of each cell.**

## Step 1 — Install required tools
Run the cell below. Wait for it to finish (the ▶ turns into a checkmark ✓).

In [ ]:
!pip install -q google-auth-oauthlib
print('Done.')

## Step 2 — Upload your credentials file

Run the cell below. A **"Choose Files"** button will appear.
Click it and upload the `client_secret.json` file.

In [ ]:
from google.colab import files
import json, pathlib

uploaded = files.upload()

if 'client_secret.json' not in uploaded:
    # Rename if the user uploaded a file with a different name
    name = list(uploaded.keys())[0]
    pathlib.Path(name).rename('client_secret.json')
    print(f'Renamed "{name}" to client_secret.json')

data = json.loads(pathlib.Path('client_secret.json').read_text())
print('File uploaded successfully.')

## Step 3 — Authorize your Google account

Run the cell below. It will print a long link.

1. **Click the link** (or copy-paste it into your browser)
2. **Sign in** with the Google account that owns the YouTube channel and the Drive folders
3. Click **Allow** on the permissions screen
4. Your browser will show an **error page** — that is completely normal and expected
5. **Copy the full URL** from your browser's address bar (the whole thing, starting with `http://localhost`)
6. Come back here and paste it in the next step

In [ ]:
from google_auth_oauthlib.flow import Flow

SCOPES = [
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/youtube',
    'https://www.googleapis.com/auth/youtube.force-ssl',
]

flow = Flow.from_client_secrets_file(
    'client_secret.json',
    scopes=SCOPES,
    redirect_uri='http://localhost',
)

auth_url, _ = flow.authorization_url(prompt='consent', access_type='offline')

print('Open this link in your browser:')
print()
print(auth_url)
print()
print('After clicking Allow, copy the full URL from the error page and run the next cell.')

## Step 4 — Paste the URL and get your credentials

Run the cell below. When prompted, paste the full URL from your browser's address bar and press **Enter**.

In [ ]:
import os
# oauthlib blocks http:// URLs by default; localhost is safe to allow
os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'

redirect_response = input('Paste the full URL from your browser here: ').strip()

# Support pasting just the code instead of the full URL
if not redirect_response.startswith('http'):
    redirect_response = 'http://localhost/?code=' + redirect_response

flow.fetch_token(authorization_response=redirect_response)
creds = flow.credentials

print()
print('=' * 60)
print('SUCCESS! Send these 3 values to your service provider:')
print('=' * 60)
print()
print('GOOGLE_CLIENT_ID')
print(creds.client_id)
print()
print('GOOGLE_CLIENT_SECRET')
print(creds.client_secret)
print()
print('GOOGLE_REFRESH_TOKEN')
print(creds.refresh_token)
print()
print('=' * 60)
print('Keep these values private. Do not share them publicly.')